### Install new libraries

In [85]:
#!pip install ddgs trafilatura

### Setup

In [86]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from pprint import pprint
import json
from pprint import pprint
from IPython.display import display, Markdown
from ddgs import DDGS
import trafilatura


# ------------------------------
# Setup
# ------------------------------

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if OPENAI_API_KEY is None:
	raise Exception("API key is missing")
client = OpenAI()

MODEL = "gpt-4.1-mini"


### Step 1: Define the Tools

In [87]:
def search_web(query: str):
	"""Search the web using DuckDuckGo browser. Returns 3 results."""
	ddgs = DDGS()
	results = ddgs.text(query, max_results=3)
	print(f"  \u2705 Got results")
	return json.dumps(results, indent=2)

In [88]:
def fetch_url(url: str):
	"""Fetch the content of a URL using trafilatura."""
	downloaded = trafilatura.fetch_url(url)
	if downloaded:
		text =  trafilatura.extract(downloaded)
		if text:
			print(f"  \u2705 Got text: {len(text)} characters")
			return text
	print(f"  \u274C Failed to fetch or extract text.") # For debugging
	return f"Could not extract text from {url}. Try a different source." # For LLM input feedback


In [89]:
#Test search_web function with a sample query
#search_web("Apples in Madagascar") #Calling the search_web function to test it with a sample query(str)

#search_web("AI in healthcare in 2030") #Calling the search_web function to test it with a sample query(str)

In [90]:
#Test fetch_url function with a sample URL
#result = fetch_url("https://explurt.com/blog-english/madagascar-english/the-list-of-fruits-to-eat-in-madagascar/") #Calling the fetch_url function to test it with a sample URL(str)
#print(result)

### Step 2: Describe as LLM tools

In [91]:
tools = []

In [92]:
#Describe search_web function as an LLM tool
search_web_function = {
	"name": "search_web",
	"description": "Search the web using DuckDuckGo browser. Returns 3 results.",
	"parameters": {
		"type": "object",
		"properties": {
			"query": {
				"type": "string",
				"description": "The search query to find relevant websites."
			}
		},
		"required": ["query"]
	}
}

#add to list of tools
tools.append({"type": "function", "function": search_web_function})

In [93]:
#Describe fetch_url function as an LLM tool
fetch_url_function = {
	"name": "fetch_url",
	"description": "Fetch and extract the main content from a web page.",
	"parameters": {
		"type": "object",
		"properties": {
			"url": {
				"type": "string",
				"description": "The URL of the web page to fetch and extract text from."
			}
		},
		"required": ["url"]
	}
}

#add to list of tools
tools.append({"type": "function", "function": fetch_url_function})


In [94]:
#Check the tools list
#tools

### Step 3: Tool Call handler

In [95]:
def handle_tool_call(tool_calls):
	tool_results = []


	for tool_call in tool_calls:  # handle every tool call the model requested
		function_name = tool_call.function.name
		args = json.loads(tool_call.function.arguments)  # arguments are JSON
		print(f"  \U0001F527 Calling function {function_name} with arguments {args}") # For future debugging

		#Route to the appropriate function based on function_name
		if function_name == "search_web":
			result = search_web(args["query"])
			content = f"Search results: {result}"

		elif function_name == "fetch_url":
			result = fetch_url(args["url"])
			content = f"Fetched URL content: {result}"
		else:
			content = f"Unknown function: {function_name}"
		tool_call_result = {
			"role": "tool",
			"content": content,
			"tool_call_id": tool_call.id
		}
		tool_results.append(tool_call_result)
	return tool_results

### Step 4: The System Prompt

 This tells the LLM who it is and how to behave. The key things:
- What its job is
- What tools it has
- What process to follow
- What output format to produce




In [96]:

# RESEARCH_AGENT_PROMPT = """You are a research agent that can search the web and fetch content from URLs.
# You have access to the following tools:
# 1. search_web(query: str) -> str: Search the web using DuckDuckGo browser. Returns 3 results.
# 2. fetch_url(url: str) -> str: Fetch and extract the main content from a web page.
# Your task is to answer questions and provide information by using these tools when necessary.\
# If you need to search for information, use the search_web tool. If you find a relevant URL, use the fetch_url tool to extract the content.\
# Always provide accurate and concise information in your responses. If you cannot find the information, respond with 'I could not find the information you requested.\
# Do not make up information. Always cite your sources when providing information from the web.\
# If you use the fetch_url tool, provide a summary of the content you extracted."""

RESEARCH_AGENT_PROMPT = """You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.

You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs
4. Reflect on what you have gathered. Do you have enough? Are there gaps?
5. If there are gaps, search again with a different query
6. When you have enough information from at least 3 different sources, synthesize into a research brief

When you are ready to deliver your final research brief, start your response with "DONE:" followed by the brief itself.

Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call — sometimes just thinking through what you have is the right move."""


### Step 5: The Agentic Loop

In [97]:
def run_research_agent(topic: str, max_iterations: int = 10) -> str:
	"""
	Run the research agent on a topic and return the research brief.
	Args:
		topic: The topic to research
		max_iterations: Safety limit to prevent infinite loops
	Returns:
		The research brief as a string
	"""
	print(f"\n🔍 Starting research on: {topic}")
	print("=" * 60)

	# Initialize conversation messages list with system prompt + research task
	messages = [
		{"role": "system", "content": RESEARCH_AGENT_PROMPT},
		{"role": "user", "content": f"Research the following topic and produce a comprehensive research brief:\n {topic}"}
	]

	# LOOP
	iteration = 0

	while iteration < max_iterations:
		iteration += 1
		print(f"\n---Iteration {iteration} ---")

		# 1. Call the LLM and get response
		response = client.chat.completions.create(
			model=MODEL,
			messages=messages,
			tools=tools,
		)

		message = response.choices[0].message
		messages.append(message)

		# 2. Check if LLM called tools
		if message.tool_calls:
			tool_results = handle_tool_call(message.tool_calls)
			messages.extend(tool_results)

		# 3. Otherwise: no tools were called, read message content
		else:
			content = message.content # No tool calls, so there's just a normal text response from the LLM
			# Check if DONE:, then return
			if content and content.startswith("DONE:"):
				research_brief = content[len("DONE:"):].strip()  # Remove "DONE:" prefix and whitespace
				print(f"  ✅ Research complete!")
				return research_brief

			# Otherwise: not yet done
			else:
				print(f"  💭 Agent is thinking...")
				pprint(content)
				# Loop continues to next iteration

		# 4. If we're entering the final iteration, force a final answer
		if iteration == max_iterations - 1:
			print("⚠️ Safety limit reached. Stopping research in next iteration.")
			messages.append({"role": "user", "content": "You have reached the maximum number of iterations. Please deliver your final research brief now, even if it is incomplete."})

	# Fallback return
	return "Research incomplete. Maximum iterations reached without finalizing brief."

### Let's Run IT!

In [98]:
#brief = run_research_agent("The impact of AI on healthcare in 2030")
#display(Markdown(f"### Research Brief on 'The impact of AI on healthcare in 2030'\n\n{brief}"))
brief = run_research_agent("How to land a job in AI research as a recent graduate")
display(Markdown(f"### Research Brief on 'How to land a job in AI research as a recent graduate'\n\n{brief}"))


🔍 Starting research on: How to land a job in AI research as a recent graduate

---Iteration 1 ---
  🔧 Calling function search_web with arguments {'query': 'how to land a job in AI research as a recent graduate'}
  ✅ Got results

---Iteration 2 ---
  🔧 Calling function fetch_url with arguments {'url': 'https://onlinedegrees.sandiego.edu/ai-research-scientist-career/'}
  ✅ Got text: 16910 characters
  🔧 Calling function fetch_url with arguments {'url': 'https://ischool.syracuse.edu/how-to-start-a-career-in-artificial-intelligence/'}
  ✅ Got text: 17417 characters
  🔧 Calling function fetch_url with arguments {'url': 'https://research.com/degrees/entry-level-hiring-trends-for-artificial-intelligence-graduates-where-new-grads-have-the-best-odds'}
  ✅ Got text: 29677 characters

---Iteration 3 ---
  ✅ Research complete!


### Research Brief on 'How to land a job in AI research as a recent graduate'

Research Brief: How to Land a Job in AI Research as a Recent Graduate

Key Facts and Statistics:
- The average AI research scientist salary as of 2024 is approximately $130,117 annually, with ranges from $50,000 to $174,000 depending on location, experience, and education level (source: onlinedegrees.sandiego.edu).
- The U.S. Bureau of Labor Statistics projects a 23% growth in computer and information research scientist roles, which include AI research scientists, through 2032 (source: onlinedegrees.sandiego.edu).
- According to a 2024 survey by the National Association of Colleges and Employers, only 38% of AI-related entry-level job postings explicitly target new graduates without prior experience (source: research.com).
- Technical AI roles at entry-level typically start between $90,000 and $130,000 in the U.S., with higher salaries in tech hubs like San Francisco and Boston (source: ischool.syracuse.edu and research.com).
- Approximately 65% of hiring managers prefer candidates with internships or project-based experience versus purely theoretical knowledge (source: research.com).

Main Themes and Arguments:
1. Educational Background and Skills: Typically, a STEM bachelor’s degree in computer science, mathematics, physics, or electrical engineering is required. Advanced degrees (Masters or PhD) improve competitiveness, especially for research-focused roles. Core technical skills include programming in Python, Java, R; mathematics (linear algebra, calculus, statistics); machine learning frameworks (TensorFlow, PyTorch); and deep learning models (CNNs, RNNs). Soft skills such as problem-solving, communication, and adaptability are equally important.
2. Practical Experience: Internships, co-op programs, open-source contributions, and participating in competitions (e.g., Kaggle) significantly increase chances of employment. Employers emphasize demonstrable hands-on experience and portfolios on platforms like GitHub showing real-world AI projects.
3. Building a Portfolio and Networking: Developing a portfolio with well-documented AI projects, including problem descriptions, data handling, model choice explanations, and results is critical. Networking by attending AI conferences, engaging on professional social networks (LinkedIn, X), and participating in hackathons aids visibility and opportunities.
4. Roles and Career Pathways: Entry-level roles include AI/ML Engineer, AI research assistant, data analyst, AI developer, and business intelligence analyst. Some roles require steep technical skills, while others emphasize cross-disciplinary applications and AI tool integration.
5. Geographical and Industry Considerations: Most jobs are concentrated in tech hubs (San Francisco, Boston, Seattle). Emerging tech regions (Austin, Raleigh) offer opportunities with less competition and lower living costs. Industries beyond tech—including finance, healthcare, manufacturing, and government research—also hire AI graduates, often requiring knowledge of domain-specific applications.
6. Overcoming Barriers: New graduates face challenges such as job requirements for prior experience, imposter syndrome, and the “black box” nature of AI models. Strategies include focusing on practical skills first, continuous learning, ethical AI awareness, and building AI-native habits (using AI tools in workflow).
7. Certifications and Alternative Learning: While degrees remain crucial, certifications like Certified Artificial Intelligence Specialist and completion of machine learning bootcamps can supplement skills. Self-directed learning via online courses (e.g., Coursera’s Andrew Ng’s Machine Learning specialization) is highly recommended.
8. AI Research Scientist vs AI Engineer: Researchers focus on advancing theory, algorithms, and publishing findings, often requiring strong academic foundations. AI engineers implement these advancements into scalable applications, needing strong programming and system deployment skills.

Notable Data Points:
- 23% projected job growth in AI research science through 2032.
- Only about 38% of entry-level AI jobs explicitly in job listings target new graduates.
- Entry-level AI salaries range broadly from $90,000 to $130,000+, influenced by region and skillset.
- Employers increasingly use AI-powered screening tools emphasizing demonstrated skills.
- Practical projects and internships can boost starting salary offers by up to 10%.
- Strong disparities exist in hiring concentration—with major tech hubs dominating but emerging cities gaining traction.

Source URLs:
1. AI Research Scientist Career Guide — University of San Diego: https://onlinedegrees.sandiego.edu/ai-research-scientist-career/
2. How to Start a Career in Artificial Intelligence — Syracuse University iSchool: https://ischool.syracuse.edu/how-to-start-a-career-in-artificial-intelligence/
3. 2027 Entry-Level Hiring Trends for Artificial Intelligence Graduates — research.com: https://research.com/degrees/entry-level-hiring-trends-for-artificial-intelligence-graduates-where-new-grads-have-the-best-odds

Summary:
To land a job in AI research as a recent graduate, one must combine strong educational credentials in STEM fields with practical experience in machine learning and AI projects. Building a portfolio, gaining internships, and developing a network within AI communities are crucial steps. Understanding the landscape of available roles, preparing to meet industry-specific skill demands, and choosing geographic areas with better hiring odds can enhance success. Pursuing additional certifications and staying adaptable to rapid AI advancements further improve employability. Though competitive, the expanding AI job market offers promising growth for prepared graduates.